# UMAP-DEA vs PCA-DEA — Side-by-Side Comparison

**How to use:** Configure Run A and Run B using the dropdowns below, then the plots will update automatically.
Compare any two runs (e.g. UMAP-DEA with N=100 vs PCA-DEA with N=100) across all dimension-reduction levels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import Dropdown, Checkbox, VBox, HBox, HTML, Layout, Output
from IPython.display import display, clear_output
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)
%matplotlib inline

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

import os

# Resolve path to all_results.csv (two levels up from this notebook in experiments/)
CSV_PATH = os.path.join(os.getcwd(), 'all_results.csv')
if not os.path.exists(CSV_PATH):
    # Fallback: navigate relative to the notebook file if run in classic Jupyter
    CSV_PATH = os.path.join(os.getcwd(), '..', 'all_results.csv')

CSV_PATH = os.path.abspath(CSV_PATH)
print(f'Loading from: {CSV_PATH}')

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} rows from all_results.csv')
print(f'Columns: {len(df.columns)}')

# Map pca column to readable labels
df['algorithm'] = df['pca'].map({True: 'UMAP-DEA', False: 'PCA-DEA'})

print(f'\nAvailable parameter values:')
print(f'  Algorithm:  {sorted(df["algorithm"].unique())}')
print(f'  N (inputs): {sorted(df["N"].unique())}')
print(f'  n (DMUs):   {sorted(df["n"].unique())}')
print(f'  RTS:        {sorted(df["rts"].unique())}')
print(f'  Gamma:      {sorted(df["gamma"].unique())}')

In [ ]:
# ============================================================
# 2. BUILD INTERACTIVE WIDGETS
# ============================================================

ALGORITHMS = sorted(df['algorithm'].unique())
N_VALUES   = sorted(df['N'].unique())
n_VALUES   = sorted(df['n'].unique())
RTS_VALUES = sorted(df['rts'].unique())
GAMMA_VALUES = sorted(df['gamma'].unique())

METRICS = {
    'MAE':                  ('mae_mean', 'mae_std'),
    'Spearman':             ('spearmanr_mean', 'spearmanr_std'),
    'Pearson':              ('pearsonr_mean', 'pearsonr_std'),
    'Kendall':              ('kendalltau_mean', 'kendalltau_std'),
    'Proportion Efficient': ('prop_efficient_mean', 'prop_efficient_std'),
    'Number Efficient':     ('nr_efficient_mean', 'nr_efficient_std'),
}

# --- Run A widgets ---
w_algo_a = Dropdown(options=ALGORITHMS, value='UMAP-DEA', description='Algorithm A:', layout=Layout(width='220px'))
w_N_a    = Dropdown(options=N_VALUES, value=100, description='N A:', layout=Layout(width='180px'))
w_n_a    = Dropdown(options=n_VALUES, value=200, description='n A:', layout=Layout(width='180px'))
w_rts_a  = Dropdown(options=RTS_VALUES, value='vrs', description='RTS A:', layout=Layout(width='180px'))
w_gamma_a= Dropdown(options=GAMMA_VALUES, value=0.5, description='γ A:', layout=Layout(width='180px'))

# --- Run B widgets ---
w_algo_b = Dropdown(options=ALGORITHMS, value='PCA-DEA', description='Algorithm B:', layout=Layout(width='220px'))
w_N_b    = Dropdown(options=N_VALUES, value=100, description='N B:', layout=Layout(width='180px'))
w_n_b    = Dropdown(options=n_VALUES, value=200, description='n B:', layout=Layout(width='180px'))
w_rts_b  = Dropdown(options=RTS_VALUES, value='vrs', description='RTS B:', layout=Layout(width='180px'))
w_gamma_b= Dropdown(options=GAMMA_VALUES, value=0.5, description='γ B:', layout=Layout(width='180px'))

# --- Shared widgets ---
w_metric = Dropdown(options=list(METRICS.keys()), value='Kendall', description='Metric:', layout=Layout(width='220px'))
w_show_std = Checkbox(value=True, description='Show ±1 std')

# Layout
panel_a = VBox([
    HTML('<b style="color:#1f77b4;">Run A</b>'),
    HBox([w_algo_a, w_N_a, w_n_a]),
    HBox([w_rts_a, w_gamma_a]),
])

panel_b = VBox([
    HTML('<b style="color:#ff7f0e;">Run B</b>'),
    HBox([w_algo_b, w_N_b, w_n_b]),
    HBox([w_rts_b, w_gamma_b]),
])

shared_controls = VBox([
    HTML('<b>Shared Controls</b>'),
    HBox([w_metric, w_show_std]),
])

ui = VBox([
    HTML('<hr style="margin:5px 0">'),
    HBox([panel_a, panel_b, shared_controls]),
    HTML('<hr style="margin:5px 0">'),
])

display(ui)

In [ ]:
# ============================================================
# 3. FILTERING & PLOTTING
# ============================================================

DIM_ORDER = ['log', 'sqrt', 'ten_percent', 'half', 'original']
DIM_LABELS = ['log(N)', '√N', '10%', 'N/2', 'N (original)']
COLOR_A = '#1f77b4'
COLOR_B = '#ff7f0e'

output = Output()


def update_plot(*args):
    """Read all widget values, filter data, produce the plot + table."""

    # --- Read current widget values ---
    algo_a = w_algo_a.value
    N_a    = w_N_a.value
    n_a    = w_n_a.value
    rts_a  = w_rts_a.value
    gamma_a = w_gamma_a.value

    algo_b = w_algo_b.value
    N_b    = w_N_b.value
    n_b    = w_n_b.value
    rts_b  = w_rts_b.value
    gamma_b = w_gamma_b.value

    metric_name = w_metric.value
    show_std    = w_show_std.value

    mean_col, std_col = METRICS[metric_name]

    output.clear_output(wait=True)

    with output:
        # --- Filter for Run A ---
        mask_a = (
            (df['algorithm'] == algo_a) &
            (df['N'] == N_a) &
            (df['n'] == n_a) &
            (df['rts'] == rts_a) &
            (df['gamma'] == gamma_a)
        )
        df_a = df[mask_a]

        # --- Filter for Run B ---
        mask_b = (
            (df['algorithm'] == algo_b) &
            (df['N'] == N_b) &
            (df['n'] == n_b) &
            (df['rts'] == rts_b) &
            (df['gamma'] == gamma_b)
        )
        df_b = df[mask_b]

        # --- Validate data ---
        if len(df_a) == 0:
            display(HTML(f'<p style="color:red;"><b>No data for Run A:</b> '
                         f'{algo_a}, N={N_a}, n={n_a}, {rts_a}, γ={gamma_a}</p>'))
            return
        if len(df_b) == 0:
            display(HTML(f'<p style="color:red;"><b>No data for Run B:</b> '
                         f'{algo_b}, N={N_b}, n={n_b}, {rts_b}, γ={gamma_b}</p>'))
            return

        # --- Align to dim_reduction_level order ---
        levels_present = set(df_a['dim_reduction_level'].unique()) & set(df_b['dim_reduction_level'].unique())
        levels_ordered = [l for l in DIM_ORDER if l in levels_present]
        labels_ordered = [DIM_LABELS[DIM_ORDER.index(l)] for l in levels_ordered]

        if len(levels_ordered) == 0:
            display(HTML('<p style="color:red;">No common dim_reduction_level values between the two runs.</p>'))
            return

        # Aggregate (mean across seeds) per dim_reduction_level
        agg_a = df_a.groupby('dim_reduction_level').agg(
            metric_mean=(mean_col, 'mean'),
            metric_std=(std_col, 'mean'),
            n_seeds=('seed', 'nunique'),
        ).reindex(levels_ordered).reset_index()

        agg_b = df_b.groupby('dim_reduction_level').agg(
            metric_mean=(mean_col, 'mean'),
            metric_std=(std_col, 'mean'),
            n_seeds=('seed', 'nunique'),
        ).reindex(levels_ordered).reset_index()

        # --- Plot ---
        fig, (ax_bar, ax_diff) = plt.subplots(2, 1, figsize=(12, 8),
                                              gridspec_kw={'height_ratios': [3, 1.5]})

        x = np.arange(len(levels_ordered))
        width = 0.35

        # Bar chart — side by side
        bars_a = ax_bar.bar(x - width/2, agg_a['metric_mean'].values, width,
                            yerr=agg_a['metric_std'].values if show_std else None,
                            capsize=5, color=COLOR_A, alpha=0.85, edgecolor='white',
                            label=f'Run A: {algo_a}')
        bars_b = ax_bar.bar(x + width/2, agg_b['metric_mean'].values, width,
                            yerr=agg_b['metric_std'].values if show_std else None,
                            capsize=5, color=COLOR_B, alpha=0.85, edgecolor='white',
                            label=f'Run B: {algo_b}')

        # Annotate bars with values
        for bar in bars_a:
            h = bar.get_height()
            if not np.isnan(h):
                ax_bar.text(bar.get_x() + bar.get_width()/2., h + 0.002,
                            f'{h:.3f}', ha='center', va='bottom', fontsize=8, color=COLOR_A)
        for bar in bars_b:
            h = bar.get_height()
            if not np.isnan(h):
                ax_bar.text(bar.get_x() + bar.get_width()/2., h + 0.002,
                            f'{h:.3f}', ha='center', va='bottom', fontsize=8, color=COLOR_B)

        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels(labels_ordered, fontsize=11)
        ax_bar.set_ylabel(metric_name, fontsize=12)
        ax_bar.set_title(
            f'{metric_name}: {algo_a} (N={N_a}, n={n_a}, {rts_a.upper()})  vs  '
            f'{algo_b} (N={N_b}, n={n_b}, {rts_b.upper()})',
            fontsize=14, fontweight='bold'
        )
        ax_bar.legend(fontsize=10, loc='best')
        ax_bar.grid(True, alpha=0.3, axis='y')

        # --- Difference panel (A − B) ---
        diff_values = agg_a['metric_mean'].values - agg_b['metric_mean'].values
        diff_colors = [COLOR_A if d >= 0 else COLOR_B for d in diff_values]
        ax_diff.bar(x, diff_values, width * 1.5, color=diff_colors, alpha=0.75, edgecolor='white')
        ax_diff.axhline(y=0, color='black', linewidth=0.8)

        for i, d in enumerate(diff_values):
            if not np.isnan(d):
                va = 'bottom' if d >= 0 else 'top'
                offset = 0.002 if d >= 0 else -0.002
                ax_diff.text(i, d + offset, f'{d:+.3f}', ha='center', va=va, fontsize=8,
                            color='black', fontweight='bold')

        ax_diff.set_xticks(x)
        ax_diff.set_xticklabels(labels_ordered, fontsize=11)
        ax_diff.set_ylabel('Δ (A − B)', fontsize=12)
        ax_diff.set_title('Difference (Run A − Run B)', fontsize=12, fontweight='bold', color='#555555')
        ax_diff.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        display(fig)
        plt.close(fig)

        # --- Summary Table ---
        table_data = {
            'Dim Reduction': labels_ordered,
            f'{algo_a} ({metric_name})': [f'{v:.5f}' if not np.isnan(v) else 'NaN' for v in agg_a['metric_mean'].values],
            f'{algo_b} ({metric_name})': [f'{v:.5f}' if not np.isnan(v) else 'NaN' for v in agg_b['metric_mean'].values],
            'Δ (A−B)': [f'{d:+.5f}' if not np.isnan(d) else 'NaN' for d in diff_values],
        }
        table_df = pd.DataFrame(table_data)

        caption = (f'Comparison Table — {metric_name}   '
                   f'(Run A seeds: {agg_a["n_seeds"].iloc[0]}   '
                   f'Run B seeds: {agg_b["n_seeds"].iloc[0]})')

        styled_table = table_df.style.set_caption(caption).set_table_styles([{
            'selector': 'caption',
            'props': [('font-size', '13px'), ('font-weight', 'bold'), ('text-align', 'left')]
        }])
        display(styled_table)


# --- Hook widget changes → update_plot ---
all_widgets = [
    w_algo_a, w_N_a, w_n_a, w_rts_a, w_gamma_a,
    w_algo_b, w_N_b, w_n_b, w_rts_b, w_gamma_b,
    w_metric, w_show_std,
]
for w in all_widgets:
    w.observe(update_plot, names='value')

# Initial render
update_plot()
display(output)